# 00_seed_historical_data

Populates the **source systems** (Azure SQL + ADLS Gen2 `raw` container) with a fiscal quarter of Contoso Tech retail activity.

- **Azure SQL**: INSERT into all retail.* tables via JDBC, using the workspace user's AAD token.
- **ADLS Gen2 `raw`**: write dated daily files in BOTH CSV and Parquet for supplier feeds and marketing exports.

This notebook is invoked by `deploy.ps1` on first deploy. Re-running is idempotent (TRUNCATEs SQL tables before inserting; overwrites ADLS files).

## Parameters

Injected by `deploy.ps1` via the Fabric `RunNotebook` job parameters. Defaults below let the notebook be run interactively in the Fabric UI for ad-hoc re-seeding.

In [ ]:
sql_server_fqdn   = ""   # e.g. contoso-retail-sql-abc123.database.windows.net
sql_database_name = "contoso_retail"
storage_account   = ""   # e.g. contosortabc12345
raw_container     = "raw"

# OneLake bronze targets (baked by deploy.ps1) for in-notebook Delta init
bronze_workspace_id = ""
bronze_lakehouse_id = ""

# OneLake silver_curated target (baked by deploy.ps1). The seed notebook
# materializes empty placeholder Delta tables here so the gold-workspace
# silver_curated shortcut lakehouse can shortcut to them at deploy time, before the
# silver_curated_* notebooks have ever run. Silver notebooks later replace
# these placeholders via overwriteSchema=true on first real run.
silver_curated_workspace_id = ""
silver_curated_lakehouse_id = ""

# Volume (fiscal quarter)
n_customers = 5_000
n_products  = 1_500
n_orders    = 50_000

# Seed window (last 90 days ending today)
import datetime
seed_end   = datetime.date.today()
seed_start = seed_end - datetime.timedelta(days=90)

random_seed = 42

## Setup

In [ ]:
import random
import uuid
import datetime
from typing import Iterator

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *  # noqa

random.seed(random_seed)

# --- stdlib stand-ins for the faker functions used below -----------------
_FIRST_NAMES = ["James","Mary","John","Patricia","Robert","Jennifer","Michael","Linda",
    "David","Elizabeth","William","Barbara","Richard","Susan","Joseph","Jessica",
    "Thomas","Sarah","Charles","Karen","Christopher","Nancy","Daniel","Lisa",
    "Matthew","Margaret","Anthony","Betty","Mark","Sandra","Donald","Ashley",
    "Steven","Kimberly","Paul","Emily","Andrew","Donna","Joshua","Michelle",
    "Kenneth","Carol","Kevin","Amanda","Brian","Melissa","George","Deborah",
    "Edward","Stephanie","Ronald","Dorothy","Timothy","Rebecca","Jason","Sharon",
    "Jeffrey","Laura","Ryan","Cynthia","Jacob","Amy","Gary","Kathleen",
    "Nicholas","Angela","Eric","Shirley","Jonathan","Brenda","Stephen","Emma",
    "Larry","Anna","Justin","Pamela","Scott","Nicole","Brandon","Samantha",
    "Frank","Katherine","Benjamin","Christine","Gregory","Helen","Samuel","Debra"]
_LAST_NAMES = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
    "Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas",
    "Taylor","Moore","Jackson","Martin","Lee","Perez","Thompson","White","Harris",
    "Sanchez","Clark","Ramirez","Lewis","Robinson","Walker","Young","Allen","King",
    "Wright","Scott","Torres","Nguyen","Hill","Flores","Green","Adams","Nelson",
    "Baker","Hall","Rivera","Campbell","Mitchell","Carter","Roberts","Gomez",
    "Phillips","Evans","Turner","Diaz","Parker","Cruz","Edwards","Collins","Reyes",
    "Stewart","Morris","Morales","Murphy","Cook","Rogers","Gutierrez","Ortiz",
    "Morgan","Cooper","Peterson","Bailey","Reed","Kelly","Howard","Ramos","Kim",
    "Cox","Ward","Richardson","Watson","Brooks","Chavez","Wood","James","Bennett"]
_COMPANIES = ["Northwind","Acme","Globex","Initech","Umbrella","Vandelay","Soylent",
    "Hooli","Pied Piper","Stark","Wayne","Wonka","Tyrell","Cyberdyne","Massive Dynamic",
    "Aperture","Black Mesa","Oscorp","LexCorp","Sterling Cooper","Pendant","Dunder",
    "Prestige","Bluth","Initrode","Strickland","Spacely","Cogswell","Planet Express"]
_EMAIL_DOMAINS = ["gmail.com","yahoo.com","outlook.com","hotmail.com","icloud.com","aol.com"]
_STREETS = ["Main","Oak","Pine","Maple","Cedar","Elm","Washington","Lake","Hill",
    "Park","Walnut","Spring","Center","Mill","Chestnut","Highland","Forest","River"]
_STREET_SUFFIX = ["St","Ave","Rd","Blvd","Ln","Dr","Way","Ct","Pl"]
_CITIES = ["Springfield","Riverside","Franklin","Greenville","Bristol","Clinton",
    "Fairview","Salem","Madison","Georgetown","Arlington","Centerville","Burlington",
    "Manchester","Dover","Newport","Oxford","Auburn","Milton","Hudson","Kingston",
    "Lancaster","Marion","Mount Vernon","Oakland","Portland","Richmond","Winchester"]
_STATES = ["AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
    "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
    "VT","VA","WA","WV","WI","WY"]

def fake_first_name():        return random.choice(_FIRST_NAMES)
def fake_last_name():         return random.choice(_LAST_NAMES)
def fake_company():           return random.choice(_COMPANIES)
def fake_free_email_domain(): return random.choice(_EMAIL_DOMAINS)
def fake_company_email():
    return f"contact@{fake_company().lower().replace(' ','')}.com"
def fake_street_address():
    return f"{random.randint(1,9999)} {random.choice(_STREETS)} {random.choice(_STREET_SUFFIX)}"
def fake_city():       return random.choice(_CITIES)
def fake_state_abbr(): return random.choice(_STATES)
def fake_zipcode():    return f"{random.randint(10000,99999)}"
def fake_numerify(pattern):
    return "".join(str(random.randint(0,9)) if ch == "#" else ch for ch in pattern)

def _coerce_date(v, fallback):
    if isinstance(v, datetime.datetime): return v.date()
    if isinstance(v, datetime.date):     return v
    if isinstance(v, str) and v.startswith("-") and v.endswith("y"):
        return datetime.date.today() - datetime.timedelta(days=int(v[1:-1])*365)
    return fallback

def fake_date_between(start_date, end_date):
    s = _coerce_date(start_date, datetime.date(2020,1,1))
    e = _coerce_date(end_date, datetime.date.today())
    return s + datetime.timedelta(days=random.randint(0, max(0,(e-s).days)))

def fake_date_time_between(start_date, end_date):
    if isinstance(start_date, datetime.datetime): s = start_date
    elif isinstance(start_date, datetime.date):   s = datetime.datetime.combine(start_date, datetime.time(0))
    elif isinstance(start_date, str) and start_date.startswith("-") and start_date.endswith("y"):
        s = datetime.datetime.now() - datetime.timedelta(days=int(start_date[1:-1])*365)
    else: s = datetime.datetime(2020,1,1)
    if isinstance(end_date, datetime.datetime): e = end_date
    elif isinstance(end_date, datetime.date):   e = datetime.datetime.combine(end_date, datetime.time(23,59,59))
    else: e = datetime.datetime.now()
    delta = max(0, int((e-s).total_seconds()))
    return s + datetime.timedelta(seconds=random.randint(0, delta))

def fake_date_of_birth(minimum_age=18, maximum_age=80):
    today = datetime.date.today()
    return today - datetime.timedelta(days=random.randint(minimum_age*365, maximum_age*365))

print(f"Seeding window: {seed_start} to {seed_end} ({(seed_end-seed_start).days} days)")
print(f"Volume: {n_customers:,} customers, {n_products:,} products, {n_orders:,} orders")
print(f"SQL: {sql_server_fqdn}/{sql_database_name}")
print(f"ADLS: abfss://{raw_container}@{storage_account}.dfs.core.windows.net/")


## Reference data (categories, brands, suppliers, warehouses, stores)

Hand-curated lists — these are the structural backbone every other table depends on.

In [ ]:
# Categories: (id, parent, name, path, sort)
CATEGORIES = [
    (1,  None, "Electronics",   "Electronics", 0),
    (2,  1,    "Smartphones",   "Electronics > Smartphones", 1),
    (3,  1,    "Laptops",       "Electronics > Laptops", 2),
    (4,  1,    "Tablets",       "Electronics > Tablets", 3),
    (5,  1,    "Headphones",    "Electronics > Headphones", 4),
    (6,  1,    "Smart Watches", "Electronics > Smart Watches", 5),
    (7,  1,    "Cameras",       "Electronics > Cameras", 6),
    (8,  1,    "Smart Home",    "Electronics > Smart Home", 7),
    (9,  1,    "Gaming",        "Electronics > Gaming", 8),
    (10, 9,    "Consoles",      "Electronics > Gaming > Consoles", 9),
    (11, 9,    "Accessories",   "Electronics > Gaming > Accessories", 10),
    (12, 1,    "Accessories",   "Electronics > Accessories", 11),
]

BRANDS = [
    (1, "Apex", "USA", True), (2, "Nimbus", "USA", True),
    (3, "Voltcraft", "Germany", False), (4, "Pixelworks", "USA", False),
    (5, "Skyline", "Korea", True), (6, "Foundry", "USA", False),
    (7, "Tundra", "Sweden", False), (8, "Kestrel", "USA", False),
    (9, "Lumen", "Japan", True), (10, "Echelon", "USA", False),
]

SUPPLIERS = [
    (i+1, f"{fake_company()} Supply", fake_company_email(), random.choice(["USA","China","Vietnam","Mexico","Korea"]), random.randint(3,30))
    for i in range(15)
]

WAREHOUSES = [
    (1, "Reno DC",       "Reno",        "NV", "USA", 250000),
    (2, "Atlanta DC",    "Atlanta",     "GA", "USA", 200000),
    (3, "Dallas DC",     "Dallas",      "TX", "USA", 180000),
    (4, "NJ DC",         "Edison",      "NJ", "USA", 150000),
    (5, "Chicago DC",    "Chicago",     "IL", "USA", 150000),
]

# 15 real US metros. lat/lon are required so the (later) weather ingest
# notebook can hit Open-Meteo without an extra geocode step.
# (id, name, type, addr, city, state, zip, country, region, lat, lon, opened, sqft, manager)
STORES = [
    (1,  "Contoso Tech – Manhattan",      "flagship", "485 5th Ave",         "New York",      "NY", "10017", "USA", "Northeast", 40.7831, -73.9712, "2018-03-15", 14000, "Mia Patel"),
    (2,  "Contoso Tech – Back Bay",       "standard", "800 Boylston St",     "Boston",        "MA", "02199", "USA", "Northeast", 42.3601, -71.0589, "2019-04-22",  8200, "Derek Hughes"),
    (3,  "Contoso Tech – Center City",    "standard", "1500 Walnut St",      "Philadelphia",  "PA", "19102", "USA", "Northeast", 39.9526, -75.1652, "2020-09-10",  7800, "Rachel Foley"),
    (4,  "Contoso Tech – Buckhead",       "standard", "3393 Peachtree Rd",   "Atlanta",       "GA", "30326", "USA", "South",     33.7490, -84.3880, "2019-11-20",  7600, "Nathan King"),
    (5,  "Contoso Tech – Brickell",       "standard", "701 Brickell Ave",    "Miami",         "FL", "33131", "USA", "South",     25.7617, -80.1918, "2021-01-12",  7400, "Carlos Diaz"),
    (6,  "Contoso Tech – The Gulch",      "standard", "1100 Broadway",       "Nashville",     "TN", "37203", "USA", "South",     36.1627, -86.7816, "2022-06-01",  7000, "Hannah Reeves"),
    (7,  "Contoso Tech – Dallas Uptown",  "flagship", "3000 Blackburn St",   "Dallas",        "TX", "75204", "USA", "South",     32.7767, -96.7970, "2018-08-10", 13500, "James Okafor"),
    (8,  "Contoso Tech – Domain",         "standard", "11410 Century Oaks",  "Austin",        "TX", "78758", "USA", "South",     30.2672, -97.7431, "2021-05-18",  7500, "Priya Sharma"),
    (9,  "Contoso Tech – Lincoln Park",   "flagship", "2110 N Clark St",     "Chicago",       "IL", "60614", "USA", "Central",   41.8781, -87.6298, "2019-06-01", 12500, "Marcus Webb"),
    (10, "Contoso Tech – North Loop",     "standard", "729 Washington Ave",  "Minneapolis",   "MN", "55401", "USA", "Central",   44.9778, -93.2650, "2022-03-15",  7100, "Olivia Brennan"),
    (11, "Contoso Tech – LoDo",           "standard", "1601 Wynkoop St",     "Denver",        "CO", "80202", "USA", "West",      39.7392, -104.9903,"2022-09-10",  7300, "Tyler Hassan"),
    (12, "Contoso Tech – Camelback",      "standard", "2502 E Camelback Rd", "Phoenix",       "AZ", "85016", "USA", "West",      33.4484, -112.0740,"2021-08-03",  7200, "Amy Russell"),
    (13, "Contoso Tech – Hollywood",      "flagship", "6801 Hollywood Blvd", "Los Angeles",   "CA", "90028", "USA", "West",      34.0522, -118.2437,"2018-11-20", 13200, "Elena Torres"),
    (14, "Contoso Tech – South Lake Union","flagship","400 Pine St",         "Seattle",       "WA", "98101", "USA", "West",      47.6062, -122.3321,"2019-02-14", 13800, "Linda Park"),
    (15, "Contoso Tech – Pearl District", "outlet",   "1234 NW Glisan St",   "Portland",      "OR", "97209", "USA", "West",      45.5152, -122.6784,"2023-05-04",  5200, "Devin Brooks"),
]

# HQ is in Redmond WA (corporate, no store_id) — used as the work location
# for all non-store employees.
HQ_CITY, HQ_STATE = "Redmond", "WA"

# Job catalog with salary bands. is_store_role=True means the role belongs
# to a physical store (one row per store * N); False means corporate/HQ.
# Salary bands are USD/year. The seeder draws actual salaries uniformly from the band.
# (id, title, department, level, min_salary, max_salary, is_store_role)
JOB_TITLES = [
    # Store roles
    (1,  "Store Manager",            "Retail",     "M2",   65000,  95000, True),
    (2,  "Assistant Store Manager",  "Retail",     "M1",   50000,  72000, True),
    (3,  "Sales Lead",               "Retail",     "IC3",  42000,  58000, True),
    (4,  "Sales Associate",          "Retail",     "IC1",  32000,  45000, True),
    (5,  "Cashier",                  "Retail",     "IC1",  28000,  38000, True),
    (6,  "Stock Associate",          "Retail",     "IC1",  30000,  40000, True),
    # Corporate / HQ
    (7,  "CEO",                      "Executive",  "Exec", 350000, 500000, False),
    (8,  "CFO",                      "Finance",    "Exec", 280000, 400000, False),
    (9,  "CTO",                      "Technology", "Exec", 280000, 400000, False),
    (10, "VP of Stores",             "Retail",     "Exec", 220000, 320000, False),
    (11, "VP of Marketing",          "Marketing",  "Exec", 200000, 300000, False),
    (12, "VP of People",             "HR",         "Exec", 200000, 300000, False),
    (13, "Director of Operations",   "Retail",     "M3",   140000, 200000, False),
    (14, "Director of Engineering",  "Technology", "M3",   160000, 230000, False),
    (15, "Director of Marketing",    "Marketing",  "M3",   135000, 195000, False),
    (16, "Director of Finance",      "Finance",    "M3",   140000, 200000, False),
    (17, "Engineering Manager",      "Technology", "M2",   120000, 175000, False),
    (18, "Marketing Manager",        "Marketing",  "M2",    95000, 140000, False),
    (19, "HR Manager",               "HR",         "M2",    85000, 125000, False),
    (20, "Software Engineer",        "Technology", "IC3",   95000, 165000, False),
    (21, "Data Analyst",             "Technology", "IC2",   75000, 115000, False),
    (22, "Accountant",               "Finance",    "IC2",   65000,  95000, False),
    (23, "HR Generalist",            "HR",         "IC2",   55000,  80000, False),
    (24, "Marketing Specialist",     "Marketing",  "IC2",   55000,  80000, False),
]

PROMOTIONS = [
    (1, "SUMMER25",    "Summer Sale 25% Off",       "percent",       25, 50),
    (2, "SAVE50",      "$50 Off Orders Over $500",  "fixed",         50, 500),
    (3, "FREESHIP",    "Free Shipping Sitewide",    "free_shipping", 0, 0),
    (4, "FALL15",      "Fall 15% Off",              "percent",       15, 0),
    (5, "HOLIDAY20",   "Holiday 20% Off",           "percent",       20, 100),
    (6, "NEWUSER10",   "New Customer 10% Off",      "percent",       10, 0),
    (7, "BFRIDAY30",   "Black Friday 30% Off",      "percent",       30, 0),
    (8, "CYBERMON",    "Cyber Monday $75 Off",      "fixed",         75, 300),
    (9, "BACK2SCHOOL", "Back to School 12% Off",    "percent",       12, 0),
    (10,"LOYALTY25",   "Loyalty Member $25 Off",    "fixed",         25, 150),
]


## Azure SQL connection

Uses an AAD access token from `notebookutils.credentials` so the JDBC connection runs as the user who triggered the notebook (the deployer, who is SQL admin).

In [ ]:
# notebookutils is pre-bound as a global in Fabric notebooks (no import required).
# Get an AAD token for Azure SQL on behalf of the user who triggered this run.
sql_token = notebookutils.credentials.getToken("https://database.windows.net/")

jdbc_url = (
    f"jdbc:sqlserver://{sql_server_fqdn}:1433;"
    f"database={sql_database_name};"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

jdbc_props = {
    "accessToken": sql_token,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

# Spark JDBC writer tuning:
#   batchsize     -> rows per JDBC executeBatch (default 1000 is way too low)
#   numPartitions -> parallel writer tasks = parallel JDBC connections to SQL
# At numPartitions=8 on F8 + GP_S_Gen5_8 we get ~8 concurrent inserts which
# is roughly where Azure SQL log throughput peaks for this workload.
JDBC_BATCH_SIZE = 10_000
JDBC_NUM_PARTS  = 8

def write_table(df, table_fqn, mode="append", num_parts=JDBC_NUM_PARTS):
    """Write a Spark DataFrame to Azure SQL via JDBC with tuned batch + parallelism."""
    (df.repartition(num_parts)
       .write
       .format("jdbc")
       .option("url", jdbc_url)
       .option("dbtable", table_fqn)
       .option("batchsize", JDBC_BATCH_SIZE)
       .option("numPartitions", num_parts)
       .options(**jdbc_props)
       .mode(mode)
       .save())
    print(f"  wrote -> {table_fqn}")

from concurrent.futures import ThreadPoolExecutor, as_completed

def write_tables_parallel(jobs, max_workers=6):
    """jobs = list of (df, table_fqn) or (df, table_fqn, num_parts). Submits
    each write_table call to a thread pool. Spark schedules each .write() as a
    separate job; the driver fires them simultaneously and Spark's scheduler
    runs them in parallel as capacity allows. Use ONLY for tables with no FK
    dependencies on each other within the same batch."""
    def _run(j):
        if len(j) == 3:
            return write_table(j[0], j[1], num_parts=j[2])
        return write_table(j[0], j[1])
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_run, j): j[1] for j in jobs}
        for f in as_completed(futs):
            f.result()
            print(f"  done: {futs[f]}")

In [ ]:
# Idempotency: DELETE child tables first (FK-respecting order).
# Use Sparks JVM gateway to call the MSSQL JDBC driver directly (already on
# the classpath because the JDBC writer above uses it). Avoids needing
# jaydebeapi/pyodbc/pymssql -- none ship in the default Fabric Spark runtime.
def exec_sql(stmts):
    gw = spark.sparkContext._gateway
    props = gw.jvm.java.util.Properties()
    props.setProperty("accessToken", sql_token)
    conn = gw.jvm.java.sql.DriverManager.getConnection(jdbc_url, props)
    try:
        stmt = conn.createStatement()
        try:
            for s in stmts:
                stmt.execute(s)
        finally:
            stmt.close()
    finally:
        conn.close()

# employees has a self-referential FK (manager_id) so we have to NULL
# manager_id before DELETE, otherwise the row order matters within the table.
exec_sql([
    "DELETE FROM retail.reviews",
    "DELETE FROM retail.returns",
    "DELETE FROM retail.shipments",
    "DELETE FROM retail.payments",
    "DELETE FROM retail.order_items",
    "DELETE FROM retail.orders",
    "UPDATE retail.employees SET manager_id = NULL",
    "DELETE FROM retail.employees",
    "DELETE FROM retail.job_titles",
    "DELETE FROM retail.inventory",
    "DELETE FROM retail.promotions",
    "DELETE FROM retail.customers",
    "DELETE FROM retail.products",
    "DELETE FROM retail.stores",
    "DELETE FROM retail.warehouses",
    "DELETE FROM retail.suppliers",
    "DELETE FROM retail.brands",
    "DELETE FROM retail.categories",
    # Reset IDENTITY counters after DELETE. DELETE does NOT reset the IDENTITY
    # seed, so on re-run new product_id / customer_id values would start at
    # last+1 (e.g. 1501+ instead of 1+). The hardcoded refs in inventory /
    # orders / order_items below (range(1, n_products+1) etc.) would then
    # violate FKs. CHECKIDENT(...,RESEED,0) makes the next insert produce 1.
    "DBCC CHECKIDENT('retail.customers',  RESEED, 0)",
    "DBCC CHECKIDENT('retail.products',   RESEED, 0)",
    "DBCC CHECKIDENT('retail.inventory',  RESEED, 0)",
    "DBCC CHECKIDENT('retail.promotions', RESEED, 0)",
    "DBCC CHECKIDENT('retail.orders',     RESEED, 0)",
    "DBCC CHECKIDENT('retail.order_items',RESEED, 0)",
    "DBCC CHECKIDENT('retail.payments',   RESEED, 0)",
    "DBCC CHECKIDENT('retail.shipments',  RESEED, 0)",
    "DBCC CHECKIDENT('retail.returns',    RESEED, 0)",
    "DBCC CHECKIDENT('retail.reviews',    RESEED, 0)",
])
print("Truncated all retail.* tables")


## Load reference data

In [ ]:
# Categories have a self-referential FK (parent_category_id -> category_id).
# Roots must land before children. Explicit schema needed because
# parent_category_id is None for every root row.
categories_schema = StructType([
    StructField("category_id",         IntegerType(), False),
    StructField("parent_category_id",  IntegerType(), True),
    StructField("category_name",       StringType(),  False),
    StructField("category_path",       StringType(),  True),
    StructField("sort_order",          IntegerType(), True),
])

roots    = [c for c in CATEGORIES if c[1] is None]
children = [c for c in CATEGORIES if c[1] is not None]

# Sequential: root rows must commit before child rows reference them.
# num_parts=1 keeps the JDBC partition order predictable for the small table.
cats_root_df = spark.createDataFrame(roots, schema=categories_schema)
write_table(cats_root_df, "retail.categories", num_parts=1)

cats_child_df = (
    spark.createDataFrame(children, schema=categories_schema)
         .coalesce(1)
         .sortWithinPartitions("parent_category_id", "category_id")
)
write_table(cats_child_df, "retail.categories", num_parts=1)

# Remaining reference tables are independent of each other. Build all 6 first,
# then fan them out as parallel JDBC writes. Each is tiny (<200 rows) so
# num_parts=1 -- don't waste Spark partitions on small tables.
brands_df = spark.createDataFrame(
    [Row(brand_id=b[0], brand_name=b[1], country_of_origin=b[2], is_premium=b[3]) for b in BRANDS]
)
supp_df = spark.createDataFrame(
    [Row(supplier_id=s[0], supplier_name=s[1], contact_email=s[2], country=s[3], lead_time_days=s[4]) for s in SUPPLIERS]
)
wh_df = spark.createDataFrame(
    [Row(warehouse_id=w[0], warehouse_name=w[1], city=w[2], state=w[3], country=w[4], capacity_units=w[5]) for w in WAREHOUSES]
)

# stores: explicit schema -- lat/lon are DECIMAL(9,6) in SQL; force DoubleType
# so JDBC handles the conversion cleanly. region is a coarse rollup used by gold marts.
stores_schema = StructType([
    StructField("store_id",      IntegerType(), False),
    StructField("store_name",    StringType(),  False),
    StructField("store_type",    StringType(),  False),
    StructField("address_line1", StringType(),  False),
    StructField("city",          StringType(),  False),
    StructField("state",         StringType(),  True),
    StructField("postal_code",   StringType(),  True),
    StructField("country",       StringType(),  False),
    StructField("region",        StringType(),  True),
    StructField("latitude",      DoubleType(),  True),
    StructField("longitude",     DoubleType(),  True),
    StructField("opened_at",     DateType(),    False),
    StructField("square_feet",   IntegerType(), True),
    StructField("manager_name",  StringType(),  True),
])
stores_df = spark.createDataFrame([
    (s[0], s[1], s[2], s[3], s[4], s[5], s[6], s[7], s[8],
     float(s[9]), float(s[10]),
     datetime.date.fromisoformat(s[11]), s[12], s[13])
    for s in STORES
], schema=stores_schema)

# job_titles reference table -- used by employees FK
job_titles_df = spark.createDataFrame(
    [Row(job_title_id=j[0], title=j[1], department=j[2], job_level=j[3],
         min_salary=float(j[4]), max_salary=float(j[5]), is_store_role=j[6])
     for j in JOB_TITLES]
)

# promotions: explicit schema -- usage_limit is None for every row
promo_schema = StructType([
    StructField("promo_code",       StringType(),    False),
    StructField("promo_name",       StringType(),    False),
    StructField("discount_type",    StringType(),    False),
    StructField("discount_value",   DoubleType(),    False),
    StructField("min_order_amount", DoubleType(),    False),
    StructField("starts_at",        TimestampType(), False),
    StructField("ends_at",          TimestampType(), False),
    StructField("usage_limit",      IntegerType(),   True),
    StructField("times_used",       IntegerType(),   False),
])
promo_rows = [
    (p[1], p[2], p[3], float(p[4]), float(p[5]),
     datetime.datetime.combine(seed_start, datetime.time(0)),
     datetime.datetime.combine(seed_end, datetime.time(23,59,59)),
     None, 0)
    for p in PROMOTIONS
]
promo_df = spark.createDataFrame(promo_rows, schema=promo_schema)

write_tables_parallel([
    (brands_df,     "retail.brands",     1),
    (supp_df,       "retail.suppliers",  1),
    (wh_df,         "retail.warehouses", 1),
    (stores_df,     "retail.stores",     1),
    (job_titles_df, "retail.job_titles", 1),
    (promo_df,      "retail.promotions", 1),
], max_workers=6)


# Read back promotion_id values. promotions has IDENTITY(1,1) but on a re-run
# the DELETE+RESEED block resets the seed to 0, so a fresh insert produces
# IDs 0..N-1 instead of 1..N. Reading back gives us the canonical IDs to use
# in the orders cell below.
promos_back = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT promotion_id, promo_code FROM retail.promotions) t")
    .options(**jdbc_props)
    .load()
    .collect())
promotion_ids = sorted(int(r["promotion_id"]) for r in promos_back)
assert len(promotion_ids) == len(PROMOTIONS), f"expected {len(PROMOTIONS)} promotions, got {len(promotion_ids)}"
print(f"resolved {len(promotion_ids)} promotion_id values from SQL: {promotion_ids}")


## Employees

Build a synthetic org chart against `STORES` + `JOB_TITLES`:

- **HQ** (corporate, `store_id` NULL): 1 CEO → 5 VPs → ~10 directors → ~14 managers → ~20 ICs (≈50 total)
- **Per store**: 1 Store Manager → 2 Assistant Managers → 3 Sales Leads → ~19 frontline ICs (≈25 each, 15 stores → ~375)
- **Total**: ~425 employees
- **Hierarchy**: `manager_id` self-FK assembled bottom-up so every non-CEO row points at an existing manager
- **Salaries**: drawn uniformly from each job's band, rounded to nearest $500
- **Tenure**: hire_date drawn over the last 8 years, ~10% terminated within the seed window (last 2 years), `is_active=False` for terminated
- All Store Managers report up to the (single) VP of Stores


In [ ]:
# ---------------------------------------------------------------------------
# Org chart: build in level order so manager_ids always reference rows that
# come earlier in the list. Cuts the self-FK problem down to a single
# bottom-up walk -- no two-pass UPDATE needed.
# ---------------------------------------------------------------------------
GENDERS = ["Female", "Male", "Non-binary"]
GENDER_WEIGHTS = [0.48, 0.48, 0.04]

# Job-title lookup helpers
JT_BY_TITLE = {j[1]: j for j in JOB_TITLES}
def jt_id(title): return JT_BY_TITLE[title][0]
def jt_band(title):
    j = JT_BY_TITLE[title]
    return (j[4], j[5])

def gen_salary(title):
    lo, hi = jt_band(title)
    raw = random.uniform(lo, hi)
    # round to nearest $500
    return round(raw / 500.0) * 500.0

def gen_hire_date(min_years_back=0, max_years_back=8):
    days_back = random.randint(min_years_back * 365, max_years_back * 365)
    return seed_end - datetime.timedelta(days=days_back)

def maybe_terminate(hire_date, term_prob=0.10):
    """Return termination_date (or None). Terminations land in the last 2 years
    of the seed window, never before hire_date."""
    if random.random() >= term_prob:
        return None
    earliest = max(hire_date, seed_end - datetime.timedelta(days=365 * 2))
    if earliest >= seed_end:
        return None
    span = (seed_end - earliest).days
    return earliest + datetime.timedelta(days=random.randint(0, span))

employees = []   # list of dicts; converted to Rows after the build
_next_eid = [1]

def add_employee(title, store_id, manager_id, min_yrs_back=0, max_yrs_back=8, term_prob=0.10):
    eid = _next_eid[0]; _next_eid[0] += 1
    first = fake_first_name()
    last  = fake_last_name()
    email = f"{first.lower()}.{last.lower()}{eid}@contoso-tech.example"
    hire  = gen_hire_date(min_yrs_back, max_yrs_back)
    term  = maybe_terminate(hire, term_prob=term_prob)
    employees.append(dict(
        employee_id=eid,
        first_name=first,
        last_name=last,
        email=email,
        phone=fake_numerify("###-###-####"),
        date_of_birth=fake_date_of_birth(minimum_age=22, maximum_age=65),
        gender=random.choices(GENDERS, weights=GENDER_WEIGHTS)[0],
        hire_date=hire,
        termination_date=term,
        job_title_id=jt_id(title),
        store_id=store_id,
        manager_id=manager_id,
        annual_salary=gen_salary(title),
        is_active=(term is None),
        created_at=datetime.datetime.combine(hire, datetime.time(9, 0)),
    ))
    return eid

# --- HQ ------------------------------------------------------------------
# CEO: longest tenure, never terminated
ceo_id = add_employee("CEO", store_id=None, manager_id=None,
                      min_yrs_back=6, max_yrs_back=12, term_prob=0.0)

# C-suite + VPs report to CEO. VP of Stores tracked separately so store mgrs can report up.
vp_stores_id = add_employee("VP of Stores", store_id=None, manager_id=ceo_id,
                            min_yrs_back=3, max_yrs_back=10, term_prob=0.05)
vp_marketing_id = add_employee("VP of Marketing", store_id=None, manager_id=ceo_id,
                               min_yrs_back=3, max_yrs_back=10, term_prob=0.05)
vp_people_id = add_employee("VP of People", store_id=None, manager_id=ceo_id,
                            min_yrs_back=3, max_yrs_back=10, term_prob=0.05)
cfo_id = add_employee("CFO", store_id=None, manager_id=ceo_id,
                      min_yrs_back=4, max_yrs_back=10, term_prob=0.05)
cto_id = add_employee("CTO", store_id=None, manager_id=ceo_id,
                      min_yrs_back=4, max_yrs_back=10, term_prob=0.05)

# Directors. Each reports to the relevant VP/Cx. (title, manager_id)
director_specs = [
    ("Director of Operations",  vp_stores_id),
    ("Director of Operations",  vp_stores_id),
    ("Director of Engineering", cto_id),
    ("Director of Engineering", cto_id),
    ("Director of Marketing",   vp_marketing_id),
    ("Director of Marketing",   vp_marketing_id),
    ("Director of Finance",     cfo_id),
    ("Director of Finance",     cfo_id),
]
director_ids = [add_employee(t, store_id=None, manager_id=m,
                             min_yrs_back=2, max_yrs_back=8, term_prob=0.08)
                for (t, m) in director_specs]

# Managers report to a same-department director.
def dirs_for(prefix):
    return [director_ids[i] for i, (t, _) in enumerate(director_specs) if t.startswith(prefix)]
eng_dirs   = dirs_for("Director of Engineering")
mkt_dirs   = dirs_for("Director of Marketing")
fin_dirs   = dirs_for("Director of Finance")
ops_dirs   = dirs_for("Director of Operations")  # noqa: F841 -- not currently used for HQ mgrs

manager_specs = [
    ("Engineering Manager", eng_dirs),
    ("Engineering Manager", eng_dirs),
    ("Engineering Manager", eng_dirs),
    ("Marketing Manager",   mkt_dirs),
    ("Marketing Manager",   mkt_dirs),
    ("HR Manager",          [vp_people_id]),
    ("HR Manager",          [vp_people_id]),
]
manager_ids_by_title = {}
for title, pool in manager_specs:
    mid = add_employee(title, store_id=None, manager_id=random.choice(pool),
                       min_yrs_back=1, max_yrs_back=6, term_prob=0.10)
    manager_ids_by_title.setdefault(title, []).append(mid)

# HQ ICs report to their function's manager.
ic_specs = [
    ("Software Engineer",     manager_ids_by_title["Engineering Manager"], 10),
    ("Data Analyst",          manager_ids_by_title["Engineering Manager"],  4),
    ("Marketing Specialist",  manager_ids_by_title["Marketing Manager"],    4),
    ("Accountant",            fin_dirs,                                      3),  # report up to a finance director (no fin mgr)
    ("HR Generalist",         manager_ids_by_title["HR Manager"],            3),
]
for title, pool, count in ic_specs:
    for _ in range(count):
        add_employee(title, store_id=None, manager_id=random.choice(pool),
                     min_yrs_back=0, max_yrs_back=5, term_prob=0.12)

# --- Stores --------------------------------------------------------------
# Per-store: 1 SM -> 2 ASM -> 3 SL -> ~19 frontline ICs.
# Store-level termination rate is higher (retail churn).
FRONTLINE_TITLES = ["Sales Associate", "Cashier", "Stock Associate"]
FRONTLINE_WEIGHTS = [0.45, 0.30, 0.25]

for s in STORES:
    sid = s[0]
    sm_id = add_employee("Store Manager", store_id=sid, manager_id=vp_stores_id,
                         min_yrs_back=1, max_yrs_back=7, term_prob=0.08)
    asm_ids = [add_employee("Assistant Store Manager", store_id=sid, manager_id=sm_id,
                            min_yrs_back=0, max_yrs_back=5, term_prob=0.12)
               for _ in range(2)]
    sl_ids  = [add_employee("Sales Lead", store_id=sid, manager_id=random.choice(asm_ids),
                            min_yrs_back=0, max_yrs_back=4, term_prob=0.15)
               for _ in range(3)]
    # frontline ICs report to a random Sales Lead or Assistant Manager
    frontline_mgr_pool = sl_ids + asm_ids
    for _ in range(19):
        title = random.choices(FRONTLINE_TITLES, weights=FRONTLINE_WEIGHTS)[0]
        add_employee(title, store_id=sid, manager_id=random.choice(frontline_mgr_pool),
                     min_yrs_back=0, max_yrs_back=4, term_prob=0.20)

print(f"Built {len(employees):,} employees "
      f"({sum(1 for e in employees if e['store_id'] is None)} HQ, "
      f"{sum(1 for e in employees if e['store_id'] is not None)} in stores, "
      f"{sum(1 for e in employees if not e['is_active'])} terminated)")

# Per-store lookup of *currently active* store employees -- used later by the
# orders cell to assign a cashier/sales-associate to each in-store order.
store_emp_pool = {}
for e in employees:
    if e['store_id'] is not None and e['is_active']:
        store_emp_pool.setdefault(e['store_id'], []).append(e['employee_id'])


In [ ]:
# employees has a self-referential FK (manager_id) so rows must commit in
# hierarchy order (CEO first, then VPs, then directors, ...). The build above
# already added rows in level order, so employee_id == insertion order.
# coalesce(1) + sortWithinPartitions guarantees JDBC sees them in that order.
employees_schema = StructType([
    StructField("employee_id",      IntegerType(),   False),
    StructField("first_name",       StringType(),    False),
    StructField("last_name",        StringType(),    False),
    StructField("email",            StringType(),    False),
    StructField("phone",            StringType(),    True),
    StructField("date_of_birth",    DateType(),      True),
    StructField("gender",           StringType(),    True),
    StructField("hire_date",        DateType(),      False),
    StructField("termination_date", DateType(),      True),
    StructField("job_title_id",     IntegerType(),   False),
    StructField("store_id",         IntegerType(),   True),
    StructField("manager_id",       IntegerType(),   True),
    StructField("annual_salary",    DoubleType(),    False),
    StructField("is_active",        BooleanType(),   False),
    StructField("created_at",       TimestampType(), False),
])
emp_rows = [(
    e["employee_id"], e["first_name"], e["last_name"], e["email"], e["phone"],
    e["date_of_birth"], e["gender"], e["hire_date"], e["termination_date"],
    e["job_title_id"], e["store_id"], e["manager_id"],
    float(e["annual_salary"]), e["is_active"], e["created_at"],
) for e in employees]

emp_df = (spark.createDataFrame(emp_rows, schema=employees_schema)
                .coalesce(1)
                .sortWithinPartitions("employee_id"))
write_table(emp_df, "retail.employees", num_parts=1)


## Products

In [ ]:
PRODUCT_TEMPLATES = [
    # (category_id, name_prefix, brand_ids, price_range, cost_frac, colors, warranty_months)
    (2, "Smartphone Pro",   [1,2,5,9], (799,1499), 0.55, ["Black","Silver","Blue"],         24),
    (3, "Laptop Ultra",     [1,4,6,8], (999,2799), 0.60, ["Space Gray","Silver"],           24),
    (4, "Tablet Air",       [1,2,5],   (449,1199), 0.55, ["Black","White","Rose"],          12),
    (5, "Headphones Studio",[3,7,9],   (149,499),  0.45, ["Black","White","Sand"],          12),
    (6, "Smart Watch",      [1,5,9],   (199,799),  0.50, ["Black","Silver","Gold"],         12),
    (7, "Mirrorless Camera",[3,4,9],   (699,2999), 0.65, ["Black"],                          24),
    (8, "Smart Speaker",    [1,2,10],  (49,349),   0.40, ["Charcoal","Sand","Forest"],      12),
    (10,"Gaming Console",   [4,6,10],  (399,599),  0.70, ["Black","White"],                  12),
    (11,"Game Controller",  [4,6,10],  (39,89),    0.35, ["Black","White","Red","Blue"],     6),
    (12,"USB-C Cable 6ft",  [3,8],     (9,29),     0.20, ["Black","White"],                  6),
    (12,"Wireless Charger", [1,3,8],   (29,79),    0.30, ["Black","White"],                  12),
    (12,"Laptop Stand",     [6,7],     (39,129),   0.30, ["Silver","Black"],                 12),
]

import math
tmpls = PRODUCT_TEMPLATES * math.ceil(n_products / len(PRODUCT_TEMPLATES))

# Explicit schema: description / dimensions_cm / discontinued_at are None for every row
products_schema = StructType([
    StructField("sku",             StringType(),  False),
    StructField("product_name",    StringType(),  False),
    StructField("description",     StringType(),  True),
    StructField("category_id",     IntegerType(), False),
    StructField("brand_id",        IntegerType(), False),
    StructField("supplier_id",     IntegerType(), False),
    StructField("list_price",      DoubleType(),  False),
    StructField("cost",            DoubleType(),  False),
    StructField("weight_kg",       DoubleType(),  True),
    StructField("dimensions_cm",   StringType(),  True),
    StructField("color",           StringType(),  True),
    StructField("model_year",      IntegerType(), True),
    StructField("upc",             StringType(),  True),
    StructField("warranty_months", IntegerType(), True),
    StructField("is_active",       BooleanType(), False),
    StructField("launched_at",     DateType(),    True),
    StructField("discontinued_at", DateType(),    True),
])

products = []
for i, tmpl in enumerate(tmpls[:n_products], start=1):
    cat_id, name_prefix, brand_ids, price_range, cost_frac, colors, warranty = tmpl
    brand_id = random.choice(brand_ids)
    color    = random.choice(colors)
    price    = round(random.uniform(*price_range), 2)
    cost     = round(price * cost_frac, 2)
    year     = random.randint(2020, seed_end.year)
    brand_nm = next(b[1] for b in BRANDS if b[0] == brand_id)
    name     = f"{brand_nm} {name_prefix} {color} ({year})"
    products.append((
        f"CT-{i:06d}", name, None, cat_id, brand_id,
        random.randint(1, len(SUPPLIERS)),
        price, cost, round(random.uniform(0.05,4.5),3),
        None, color, year,
        str(random.randint(10**11, 10**12 - 1)), warranty, True,
        fake_date_between(start_date=datetime.date(2020,1,1), end_date=seed_end),
        None,
    ))

products_df = spark.createDataFrame(products, schema=products_schema)
write_table(products_df, "retail.products")

# Read back actual product_id values. Spark repartitions on write, so the
# IDENTITY values SQL Server assigns do NOT line up with our Python list
# order. Downstream cells (inventory / order_items) must look up by sku.
products_back = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT product_id, sku FROM retail.products) t")
    .options(**jdbc_props)
    .load()
    .collect())
sku_to_pid = {r["sku"]: int(r["product_id"]) for r in products_back}
product_ids = list(sku_to_pid.values())
assert len(product_ids) == n_products, f"expected {n_products} products, got {len(product_ids)}"
print(f"resolved {len(product_ids):,} product_id values from SQL")


## Customers

In [ ]:
loyalty_tiers   = ["Bronze","Silver","Gold","Platinum"]
loyalty_weights = [0.60, 0.25, 0.12, 0.03]

# Explicit schema: address_line2 is None for every row
customers_schema = StructType([
    StructField("email",            StringType(),    False),
    StructField("first_name",       StringType(),    False),
    StructField("last_name",        StringType(),    False),
    StructField("phone",            StringType(),    True),
    StructField("date_of_birth",    DateType(),      True),
    StructField("segment_id",       IntegerType(),   True),
    StructField("address_line1",    StringType(),    True),
    StructField("address_line2",    StringType(),    True),
    StructField("city",             StringType(),    True),
    StructField("state",            StringType(),    True),
    StructField("postal_code",      StringType(),    True),
    StructField("country",          StringType(),    True),
    StructField("loyalty_tier",     StringType(),    True),
    StructField("loyalty_points",   IntegerType(),   True),
    StructField("marketing_opt_in", BooleanType(),   True),
    StructField("created_at",       TimestampType(), False),
    StructField("last_login_at",    TimestampType(), True),
    StructField("is_active",        BooleanType(),   False),
])

def gen_customer(i):
    first = fake_first_name()
    last  = fake_last_name()
    eu    = uuid.uuid4().hex[:8]
    created = fake_date_time_between(start_date="-5y", end_date=seed_start)
    return (
        f"{first.lower()}.{last.lower()}.{eu}@{fake_free_email_domain()}",
        first, last,
        fake_numerify("###-###-####"),
        fake_date_of_birth(minimum_age=18, maximum_age=80),
        random.choices([1,2,3,4], weights=[60,25,10,5])[0],
        fake_street_address(), None,
        fake_city(), fake_state_abbr(), fake_zipcode(), "USA",
        random.choices(loyalty_tiers, weights=loyalty_weights)[0],
        random.randint(0,50000),
        random.random() < 0.55,
        created,
        fake_date_time_between(start_date=created, end_date=seed_end) if random.random() < 0.8 else None,
        True,
    )

customers = [gen_customer(i) for i in range(n_customers)]
cust_df = spark.createDataFrame(customers, schema=customers_schema)
write_table(cust_df, "retail.customers")

# Read back customer_id values for downstream FK use (orders / reviews).
customers_back = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT customer_id FROM retail.customers) t")
    .options(**jdbc_props)
    .load()
    .collect())
customer_ids = [int(r["customer_id"]) for r in customers_back]
assert len(customer_ids) == n_customers, f"expected {n_customers} customers, got {len(customer_ids)}"
print(f"resolved {len(customer_ids):,} customer_id values from SQL")


## Inventory

In [ ]:
inv_rows = []
for pid in product_ids:
    for wh in WAREHOUSES:
        inv_rows.append(Row(
            product_id=pid, location_type="warehouse", location_id=wh[0],
            quantity_on_hand=random.randint(0,500), quantity_reserved=0,
            reorder_point=random.randint(5,50), reorder_quantity=random.randint(25,200),
            last_restocked_at=fake_date_time_between(start_date=seed_start, end_date=seed_end),
        ))
inv_df = spark.createDataFrame(inv_rows)
write_table(inv_df, "retail.inventory")

## Orders + items + payments + shipments

We need real surrogate ids back from SQL for FK chaining (orders.order_id, order_items.order_item_id). Strategy:

1. INSERT orders, then SELECT them back keyed by `order_number` to get assigned `order_id`s
2. Build items / payments / shipments using those ids
3. INSERT items / payments / shipments
4. SELECT items back to get assigned `order_item_id`s for returns / reviews

In [ ]:
channels         = ["online","store","mobile"]
channel_weights  = [0.60, 0.20, 0.20]
payment_methods  = ["credit_card","debit_card","paypal","apple_pay","google_pay","store_credit","gift_card"]
payment_weights  = [0.45,0.20,0.15,0.08,0.05,0.04,0.03]
card_brands      = ["Visa","Mastercard","Amex","Discover",None,None,None]
carriers         = ["UPS","FedEx","USPS","DHL"]
order_statuses   = ["delivered","delivered","delivered","shipped","paid","cancelled"]
status_weights   = [55,15,5,10,10,5]

product_prices = {p[0]: p[6] for p in products}  # sku=idx0, list_price=idx6 (products is list of tuples for explicit schema)
skus = list(product_prices.keys())

orders_in = []  # holds (order_number, planned items) for later
for i in range(n_orders):
    on  = f"ORD-{i+1:010d}"
    cust_id = random.choice(customer_ids)
    channel = random.choices(channels, weights=channel_weights)[0]
    store_id = random.randint(1, len(STORES)) if channel == "store" else None
    promo_id = random.choice([None,None,None] + promotion_ids)
    odate = fake_date_time_between(start_date=seed_start, end_date=seed_end)
    n_items = random.choices([1,2,3,4,5], weights=[40,30,15,10,5])[0]

    items = []
    subtotal = 0.0; discount = 0.0
    for _ in range(n_items):
        sku = random.choice(skus)
        list_price = product_prices[sku]
        qty = random.randint(1,3)
        unit_price = round(list_price * random.uniform(0.85, 1.0), 2)
        line_gross = round(unit_price * qty, 2)
        line_disc  = round(line_gross * (0.1 if promo_id else 0), 2)
        line_total = round(line_gross - line_disc, 2)
        subtotal += line_gross; discount += line_disc
        items.append((sku, qty, unit_price, line_disc, line_total, random.randint(1, len(WAREHOUSES))))

    tax = round((subtotal - discount) * 0.08, 2)
    shipping = 0.0 if (subtotal - discount) > 99 else round(random.uniform(5.99, 14.99), 2)
    total = round(subtotal - discount + tax + shipping, 2)
    status = random.choices(order_statuses, weights=status_weights)[0]

    orders_in.append({
        "order_number": on, "customer_id": cust_id, "order_date": odate,
        "order_status": status, "channel": channel, "store_id": store_id,
        "subtotal": round(subtotal,2), "tax_amount": tax, "shipping_amount": shipping,
        "discount_amount": round(discount,2), "total_amount": total, "currency": "USD",
        "promotion_id": promo_id,
        "ship_address_line1": fake_street_address(), "ship_city": fake_city(),
        "ship_state": fake_state_abbr(), "ship_postal_code": fake_zipcode(), "ship_country": "USA",
        "_items": items,
    })

print(f"Prepared {len(orders_in):,} orders in driver memory")

In [ ]:
# Insert orders (without items yet)
orders_only = [{k:v for k,v in o.items() if k != "_items"} for o in orders_in]
orders_df = spark.createDataFrame([Row(**o) for o in orders_only])
write_table(orders_df, "retail.orders")

# Read back to get the IDENTITY-assigned order_id
order_ids_df = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT order_id, order_number FROM retail.orders) t")
    .options(**jdbc_props)
    .load())
order_id_map = {r['order_number']: r['order_id'] for r in order_ids_df.collect()}
print(f"Resolved {len(order_id_map):,} order_id values")

In [ ]:
# Now build items / payments / shipments using the resolved order_ids.
# sku_to_pid was populated from the products read-back in cell 16, so the
# mapping reflects the IDENTITY values SQL Server actually assigned.

item_rows = []; pay_rows = []; ship_rows = []
for o in orders_in:
    oid = order_id_map[o['order_number']]
    for (sku, qty, unit_price, line_disc, line_total, wh_id) in o['_items']:
        item_rows.append(Row(order_id=oid, product_id=sku_to_pid[sku], quantity=qty,
                             unit_price=unit_price, line_discount=line_disc,
                             line_total=line_total, fulfillment_warehouse_id=wh_id))
    # Payment
    pm = random.choices(payment_methods, weights=payment_weights)[0]
    cb = random.choice(card_brands) if "card" in pm else None
    cl4 = fake_numerify("####") if cb else None
    if o['order_status'] == "cancelled":
        pay_status = "voided"
    else:
        r = random.random()
        pay_status = "failed" if r < 0.03 else ("declined" if r < 0.04 else "captured")
    pay_rows.append(Row(order_id=oid, payment_method=pm, card_brand=cb, card_last_four=cl4,
                        amount=o['total_amount'], status=pay_status,
                        transaction_ref=uuid.uuid4().hex.upper()[:20], processed_at=o['order_date']))
    # Shipment for non-store, non-cancelled
    if o['channel'] != "store" and o['order_status'] not in ("cancelled","paid"):
        carrier = random.choice(carriers)
        tracking = f"{carrier[:3].upper()}{uuid.uuid4().hex[:12].upper()}"
        wh_id   = random.randint(1, len(WAREHOUSES))
        shipped = fake_date_time_between(start_date=o['order_date'], end_date=seed_end) if o['order_status'] in ("shipped","delivered") else None
        est_del = (shipped.date() + datetime.timedelta(days=random.randint(2,7))) if shipped else None
        delivered = fake_date_time_between(start_date=shipped, end_date=seed_end) if (o['order_status']=="delivered" and shipped) else None
        ship_status = {"delivered":"delivered","shipped":"in_transit","paid":"label_created"}.get(o['order_status'], "label_created")
        ship_rows.append(Row(order_id=oid, warehouse_id=wh_id, carrier=carrier, tracking_number=tracking,
                             shipped_at=shipped, estimated_delivery=est_del, delivered_at=delivered, status=ship_status))

print(f"Built {len(item_rows):,} items, {len(pay_rows):,} payments, {len(ship_rows):,} shipments")

In [ ]:
items_df    = spark.createDataFrame(item_rows)
payments_df = spark.createDataFrame(pay_rows)
shipments_df= spark.createDataFrame(ship_rows)

# Independent of each other (all FK only to orders, which is already written).
# Fire all 3 JDBC writes in parallel -- this is the longest single phase
# (~150k items) so the parallelism really matters here.
write_tables_parallel([
    (items_df,    "retail.order_items"),
    (payments_df, "retail.payments"),
    (shipments_df,"retail.shipments"),
], max_workers=3)

## Returns + Reviews

Generated AFTER items are committed so we can resolve IDENTITY-assigned `order_item_id` values from SQL.

- **Returns**: ~5% of order_items, refund proportional to returned quantity, status weighted toward `refunded`
- **Reviews**: ~30% of orders, one review per order on a random product, rating skewed positive (50% 5-star), `is_verified_purchase=True`

In [ ]:
# Read back items so we have IDENTITY-assigned order_item_id values.
items_back = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT order_item_id, order_id, product_id, quantity, line_total FROM retail.order_items) t")
    .options(**jdbc_props)
    .load()).collect()
print(f"Resolved {len(items_back):,} order_item_id values")

# order_id -> (customer_id, order_date) lookup from the in-memory orders we built
order_meta = {order_id_map[o["order_number"]]: (o["customer_id"], o["order_date"]) for o in orders_in}

# --------------------------------------------------------------------------
# RETURNS: ~5% of items
# --------------------------------------------------------------------------
return_reasons = ["wrong_size","damaged","not_as_described","no_longer_needed",
                  "defective","wrong_item","quality_issue","arrived_late"]
# CHECK constraint allows: requested / received / refunded / rejected.
# Weight toward refunded so the demo has interesting refund_amount totals.
return_status_pool = ["refunded","refunded","refunded","received","requested","rejected"]

ret_rows = []
for it in items_back:
    if random.random() >= 0.05:
        continue
    cust_id, odate = order_meta[it["order_id"]]
    ret_qty = random.randint(1, int(it["quantity"]))
    refund  = round(float(it["line_total"]) * (ret_qty / int(it["quantity"])), 2)
    status  = random.choice(return_status_pool)
    requested = fake_date_time_between(start_date=odate, end_date=seed_end)
    completed = fake_date_time_between(start_date=requested, end_date=seed_end) if status == "refunded" else None
    ret_rows.append(Row(
        order_id=int(it["order_id"]),
        order_item_id=int(it["order_item_id"]),
        customer_id=int(cust_id),
        return_reason=random.choice(return_reasons),
        quantity=int(ret_qty),
        refund_amount=float(refund),
        return_status=status,
        requested_at=requested,
        completed_at=completed,
    ))

# --------------------------------------------------------------------------
# REVIEWS: ~30% of orders, one review per order on a random item
# --------------------------------------------------------------------------
items_by_order = {}
for it in items_back:
    items_by_order.setdefault(it["order_id"], []).append(it)

# Rating distribution skewed positive (typical retail review pattern).
rating_weights = [3, 7, 15, 25, 50]  # for ratings 1..5
review_titles = {
    5: ["Excellent!","Love it","Highly recommend","Perfect","Worth every penny"],
    4: ["Pretty good","Solid choice","Mostly happy","Good value","Recommended"],
    3: ["It's okay","Average","Met expectations","Decent","Not bad"],
    2: ["Disappointed","Below expectations","Wouldn't buy again","Had issues","Mediocre"],
    1: ["Terrible","Do not buy","Returned it","Worst purchase","Awful"],
}
review_text = {
    5: "Exceeded my expectations -- showed up fast and quality is great.",
    4: "Good product overall. A couple minor nits but I'd buy again.",
    3: "Works as advertised but nothing special. Fair for the price.",
    2: "Build quality was below what I expected. Not great.",
    1: "Defective on arrival. Wasted my time.",
}

rev_rows = []
for oid, its in items_by_order.items():
    if random.random() >= 0.30:
        continue
    pick = random.choice(its)
    cust_id, odate = order_meta[oid]
    rating = random.choices([1,2,3,4,5], weights=rating_weights)[0]
    rev_rows.append(Row(
        product_id=int(pick["product_id"]),
        customer_id=int(cust_id),
        order_id=int(oid),
        rating=int(rating),
        review_title=random.choice(review_titles[rating]),
        review_text=review_text[rating],
        helpful_count=int(random.randint(0, 50)),
        created_at=fake_date_time_between(start_date=odate, end_date=seed_end),
        is_verified_purchase=True,
    ))

print(f"Built {len(ret_rows):,} returns, {len(rev_rows):,} reviews")

returns_df = spark.createDataFrame(ret_rows)
reviews_df = spark.createDataFrame(rev_rows)
write_tables_parallel([
    (returns_df, "retail.returns"),
    (reviews_df, "retail.reviews"),
], max_workers=2)


## ADLS Gen2: dated daily files (CSV + Parquet) in `raw/`

Two source feeds:

- `raw/supplier_inventory/YYYY-MM-DD.parquet` — daily supplier on-hand snapshot
- `raw/marketing_campaigns/YYYY-MM-DD.csv` — daily marketing campaign performance

These mimic vendor / partner data drops landing in your data lake.

In [ ]:
raw_base = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net"

# Supplier inventory: 1 row per (supplier, sku-group) per day.
# Build ALL days into a single DataFrame and write once with partitionBy. Spark
# parallelizes the per-partition write -- one Spark job with N concurrent
# tasks instead of N sequential .write() calls (each carrying its own task
# scheduling + commit overhead).
supplier_skus = [s for s in skus[:200]]  # subset that suppliers actually quote on
n_days = (seed_end - seed_start).days + 1

rows = []
for d_offset in range(n_days):
    day = seed_start + datetime.timedelta(days=d_offset)
    day_iso = day.isoformat()
    for sku in supplier_skus:
        for supp_id in range(1, len(SUPPLIERS)+1):
            if random.random() < 0.4:  # not every supplier quotes every sku
                rows.append(Row(
                    snapshot_date=day_iso,  # string keeps partition dir names clean
                    sku=sku, supplier_id=supp_id,
                    qty_available=random.randint(0, 2000),
                    unit_cost=round(random.uniform(5, 800), 2),
                    lead_time_days=random.randint(3, 30),
                ))

supplier_df = spark.createDataFrame(rows).repartition(n_days, "snapshot_date")
(supplier_df.write
    .mode("overwrite")
    .partitionBy("snapshot_date")
    .parquet(f"{raw_base}/supplier_inventory"))

print(f"Wrote supplier_inventory parquet for {n_days} days ({len(rows):,} rows total)")

In [ ]:
# Marketing campaigns: 1 row per (day, channel). Same partitionBy strategy as
# supplier_inventory -- build all days once, write once partitioned by date.
campaign_channels = ["google_ads","meta_ads","tiktok","email","influencer","display"]

rows = []
for d_offset in range(n_days):
    day = seed_start + datetime.timedelta(days=d_offset)
    day_iso = day.isoformat()
    for ch in campaign_channels:
        impressions = random.randint(5_000, 250_000)
        clicks      = int(impressions * random.uniform(0.005, 0.04))
        spend       = round(impressions / 1000 * random.uniform(2.5, 18.0), 2)
        conversions = int(clicks * random.uniform(0.01, 0.08))
        rows.append(Row(
            campaign_date=day_iso,
            channel=ch,
            impressions=impressions, clicks=clicks,
            spend_usd=spend, conversions=conversions,
        ))

mkt_df = spark.createDataFrame(rows).repartition(n_days, "campaign_date")
(mkt_df.write
    .mode("overwrite")
    .option("header", True)
    .partitionBy("campaign_date")
    .csv(f"{raw_base}/marketing_campaigns"))

print(f"Wrote marketing_campaigns csv for {n_days} days ({len(rows):,} rows total)")

## Done

Azure SQL is populated with a full fiscal quarter; ADLS `raw/` has 90+ days of supplier Parquet + marketing CSV files.

`deploy.ps1` will now set up the SQL Mirror and the ADLS Shortcut against this populated data.

In [ ]:
# Materialize empty weather Delta in bronze so silver_raw can shortcut to it.
# weather notebook (ingest_weather) later overwrites this with real data.
# Done here -- inside an already-warm Spark session -- to avoid a separate 60s notebook run.
if bronze_workspace_id and bronze_lakehouse_id:
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
    wx_schema = StructType([
        StructField('date',                 DateType(),    False),
        StructField('store_id',             IntegerType(), False),
        StructField('city',                 StringType(),  False),
        StructField('state',                StringType(),  False),
        StructField('latitude',             DoubleType(),  False),
        StructField('longitude',            DoubleType(),  False),
        StructField('temperature_max_c',    DoubleType(),  True),
        StructField('temperature_min_c',    DoubleType(),  True),
        StructField('temperature_max_f',    DoubleType(),  True),
        StructField('temperature_min_f',    DoubleType(),  True),
        StructField('precipitation_mm',     DoubleType(),  True),
        StructField('snowfall_cm',          DoubleType(),  True),
        StructField('wind_speed_max_kmh',   DoubleType(),  True),
        StructField('weather_code',         IntegerType(), True),
        StructField('weather_description',  StringType(),  True),
    ])
    wx_path = f'abfss://{bronze_workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/dbo/weather'
    (spark.createDataFrame([], wx_schema)
        .write.format('delta').mode('ignore').save(wx_path))
    print('materialized empty weather delta at', wx_path)
else:
    print('bronze workspace/lakehouse params not set; skipping weather init')


In [ ]:
# Materialize empty placeholder Delta tables in silver_curated so the
# gold-workspace silver_curated shortcut lakehouse can shortcut to them BEFORE the
# silver_curated_* notebooks have ever run. Single _placeholder column is
# enough -- silver notebooks use overwriteSchema=true and will replace the
# schema on first real run; gold sprocs use DROP + CTAS so they refresh
# downstream every run.
if silver_curated_workspace_id and silver_curated_lakehouse_id:
    from pyspark.sql.types import StructType, StructField, StringType
    placeholder = StructType([StructField('_placeholder', StringType(), True)])
    curated_base = f'abfss://{silver_curated_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_curated_lakehouse_id}/Tables/dbo'
    curated_tables = [
        'customer', 'product', 'order', 'order_line',
        'store', 'employee',
        'weather_daily',
        'session', 'session_event',
        'supplier', 'warehouse', 'promotion',
        'inventory', 'payment', 'return', 'review', 'shipment',
    ]
    for _t in curated_tables:
        (spark.createDataFrame([], placeholder)
            .write.format('delta').mode('ignore').save(f'{curated_base}/{_t}'))
    print(f'materialized {len(curated_tables)} placeholder silver_curated deltas under {curated_base}')
else:
    print('silver_curated workspace/lakehouse params not set; skipping silver_curated init')


In [ ]:
notebookutils.notebook.exit("OK")
